<a href="https://colab.research.google.com/github/Zeshan811/StarterNotebookA1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'dim_content': f"read_parquet('{RE
                                    L}/dim_content.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


In [2]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position_month,
        (SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)) AS ctr_month
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

print(feature_frame.shape)
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 6)


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position_month,ctr_month
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,0.001073
1,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,6.481453,0.000000
2,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,2.987198,0.000000
3,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,0.001066
4,content_05434271b257bb68,client_73cda7b4e4f265ea,1421.0,6.0,6.320337,0.004222


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



My Week-5 model used a plain random train/test split — but content items
from the same client can share patterns (similar site structure, similar
audience), so a random split could let the model "cheat" by seeing
similar pages from the same client in both train and test. Here I re-run
the same K-Means setup under a **client-grouped split** (whole clients
held out, never split across train/test) and compare stability before
vs after.

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = ["total_impressions", "total_clicks", "avg_position_month", "ctr_month"]
X = feature_frame[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# BEFORE: random split (ignores client grouping) — same as Week 5
X_train_random, X_test_random = train_test_split(X_scaled, test_size=0.3, random_state=42)

km_random = KMeans(n_clusters=6, random_state=42, n_init=10).fit(X_train_random)
random_train_sil = silhouette_score(X_train_random, km_random.labels_)
random_test_labels = km_random.predict(X_test_random)
random_test_sil = silhouette_score(X_test_random, random_test_labels)

print(f"BEFORE (random split) — train silhouette: {random_train_sil:.3f}, test silhouette: {random_test_sil:.3f}")

BEFORE (random split) — train silhouette: 0.487, test silhouette: 0.480


In [4]:
groups = feature_frame["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_scaled, groups=groups))

X_train_grouped, X_test_grouped = X_scaled[train_idx], X_scaled[test_idx]

# Confirm no client overlap
train_clients = set(groups[train_idx])
test_clients = set(groups[test_idx])
print("Client overlap between train/test:", len(train_clients & test_clients), "(should be 0)")

km_grouped = KMeans(n_clusters=6, random_state=42, n_init=10).fit(X_train_grouped)
grouped_train_sil = silhouette_score(X_train_grouped, km_grouped.labels_)
grouped_test_labels = km_grouped.predict(X_test_grouped)
grouped_test_sil = silhouette_score(X_test_grouped, grouped_test_labels)

print(f"AFTER (client-grouped split) — train silhouette: {grouped_train_sil:.3f}, test silhouette: {grouped_test_sil:.3f}")

Client overlap between train/test: 0 (should be 0)
AFTER (client-grouped split) — train silhouette: 0.475, test silhouette: 0.490


| Split type | Train silhouette | Test silhouette |
|---|---|---|
| BEFORE — random split | 0.487 | 0.480 |
| AFTER — client-grouped split | 0.475 | 0.490 |

(Fill in: did the grouped split reveal the random split was overly
optimistic? Or did the score hold up either way? Either answer is a
valid, honest finding — say which happened and why it makes sense given
that content items belong to specific clients.)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


Repeating the Week-3 leakage hunt on my final feature set, one more time,
now that the model is built.

In [5]:
print("LEAKAGE AUDIT — final feature set:", features)
print()
print("1. Are any features calculated after the decision point?")
print("   No — all 4 features are aggregated only from March 2026, the")
print("   single month being clustered. No later month's data is touched.")
print()
print("2. Does the feature window overlap the target window?")
print("   N/A for clustering — there is no separate target window, since")
print("   clustering has no label. All features come from the same,")
print("   single, clearly-bounded window.")
print()
print("3. Did any FlyRank product output (health_score, priority_score,")
print("   action_type) slip in as a feature?")
print("   No —", set(features) & {"health_score", "priority_score", "action_type"}, "(should be empty set)")
print()
print("4. Does a derived field secretly encode information from outside")
print("   the observed window?")
print("   No — ctr_month is derived only from impressions/clicks within")
print("   the same March 2026 window, not from any other period.")
print()
print("5. Are duplicate/related rows split across train and test unfairly?")
print("   Fixed in Section 2 — the grouped split now keeps each client's")
print("   content entirely in train OR test, never split across both.")

LEAKAGE AUDIT — final feature set: ['total_impressions', 'total_clicks', 'avg_position_month', 'ctr_month']

1. Are any features calculated after the decision point?
   No — all 4 features are aggregated only from March 2026, the
   single month being clustered. No later month's data is touched.

2. Does the feature window overlap the target window?
   N/A for clustering — there is no separate target window, since
   clustering has no label. All features come from the same,
   single, clearly-bounded window.

3. Did any FlyRank product output (health_score, priority_score,
   action_type) slip in as a feature?
   No — set() (should be empty set)

4. Does a derived field secretly encode information from outside
   the observed window?
   No — ctr_month is derived only from impressions/clicks within
   the same March 2026 window, not from any other period.

5. Are duplicate/related rows split across train and test unfairly?
   Fixed in Section 2 — the grouped split now keeps each client'

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My original claim (from Week 5/7 notebook):**
"K-Means finds 6 distinct archetypes, and position/impression scale drive
most of the cluster separation."

**Rewritten in safe language:**
"Under an observed slice of March 2026 data, K-Means grouping on 4 GSC-
based features produced 6 clusters whose average feature profiles
differ noticeably — most visibly along average position and impression
scale. This is a directional pattern in this dataset and validation
setup, not a claim about SEO in general; it should be treated as
decision-support for prioritizing review, not a guarantee that any given
page belongs permanently to one archetype."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.